In [1]:
!pip install -U transformers==4.57.1 peft==0.20.0 datasets==4.3.0 accelerate==1.11.0 bitsandbytes==0.48.2 trl==0.23.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.7 MB/s eta 0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling 

In [2]:
import torch
import importlib.metadata as md

print("Torch:", torch.__version__)
print("TorchVision:", md.version("torchvision"))
print("Transformers:", md.version("transformers"))
print("PEFT:", md.version("peft"))
print("Accelerate:", md.version("accelerate"))
print("TRL:", md.version("trl"))

Torch: 2.10.0+cu128
TorchVision: 0.25.0+cu128
Transformers: 4.57.1
PEFT: 0.20.0
Accelerate: 1.11.0
TRL: 0.23.0


In [3]:
import sys
import torch
import transformers
import peft
import datasets
import accelerate
import bitsandbytes
import trl

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
Transformers: 4.57.1
PEFT: 0.20.0
Datasets: 4.3.0
Accelerate: 1.11.0
bitsandbytes: 0.48.2
TRL: 0.23.0
CUDA available: True
GPU: Tesla T4
CUDA: 12.8


In [4]:
from datasets import load_dataset

In [5]:
dataset=load_dataset(
    "json",
    data_files="/kaggle/input/datasets/shouldichangemyname/phi2cot/cot_trainable.json"
)
print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 31705
    })
})
{'text': '[Question] A few players are playing a boardgame. The current state of the game is as follows. The cat has a hot chocolate. The cat learns the basics of resource management from the jellyfish. The parrot sings a victory song for the jellyfish. And the rules of the game are as follows. Rule1: If the parrot sings a victory song for the jellyfish and the cat learns the basics of resource management from the jellyfish, then the jellyfish becomes an enemy of the canary. Rule2: Regarding the cat, if it has something to drink, then we can conclude that it attacks the green fields of the swordfish. Rule3: If you are positive that you saw one of the animals becomes an enemy of the canary, you can be certain that it will also learn elementary resource management from the baboon. Rule4: If the jellyfish learns the basics of resource management from the baboon, then the baboon is not going to

In [6]:
print(dataset["train"].shape)

(31705, 1)


In [7]:
from transformers import AutoTokenizer,AutoModelForCausalLM

In [8]:
model_name='microsoft/phi-2'

In [9]:
tokenizer=AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [10]:
if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token

In [11]:
print("Tokenizer loaded")
print("Vocab size",tokenizer.vocab_size)
print("EOS Token",tokenizer.eos_token)

Tokenizer loaded
Vocab size 50257
EOS Token <|endoftext|>


In [12]:
import torch
from transformers import BitsAndBytesConfig

In [13]:
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [14]:
print("phi2 model loaded successfully")
print("Model device", base_model.device)

phi2 model loaded successfully
Model device cuda:0


In [15]:
from peft import LoraConfig, get_peft_model,prepare_model_for_kbit_training

base_model=prepare_model_for_kbit_training(base_model)

lora_config=LoraConfig(
    r=64,
    lora_alpha=16,
    bias="none",
    lora_dropout=0.1,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "dense",
        "fc1",
        "fc2"
    ]
)

peft_model=get_peft_model(base_model,lora_config)
peft_model.print_trainable_parameters()

trainable params: 94,371,840 || all params: 2,874,055,680 || trainable%: 3.2836


In [16]:
from transformers import TrainingArguments

In [17]:
training_args=TrainingArguments(
    output_dir='/kaggle/working/models/phi2-cot',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    weight_decay=0.001,
    optim="paged_adamw_32bit",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    fp16=False,
    bf16=False,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

print("Training config created")

Training config created


In [18]:
from trl import SFTTrainer

In [19]:
trainer=SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/31705 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/31705 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2049 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/31705 [00:00<?, ? examples/s]

In [20]:
print("SFT trainer created")

SFT trainer created


In [21]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture 

Step,Training Loss
10,1.803300
20,1.613000
30,1.735100
40,1.696200
50,1.563300
60,1.576900
70,1.780700
80,1.502700
90,1.698300
100,1.463200


TrainOutput(global_step=31705, training_loss=1.6149171330202756, metrics={'train_runtime': 35017.5413, 'train_samples_per_second': 0.905, 'train_steps_per_second': 0.905, 'total_flos': 2.0853715561316352e+17, 'train_loss': 1.6149171330202756, 'entropy': 1.6092838287353515, 'num_tokens': 12670944.0, 'mean_token_accuracy': 0.6095524787902832, 'epoch': 1.0})

In [24]:
print("Done training")

Done training


In [26]:
import shutil

shutil.make_archive(
    "/kaggle/working/phi2-cot-final",
    "zip",
    root_dir="/kaggle/working/models/phi2-cot",
    base_dir="checkpoint-31705"
)

print("Created: /kaggle/working/phi2-cot-final.zip")

Created: /kaggle/working/phi2-cot-final.zip
